In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import hankel2
plt.rcParams.update({
    "font.size" : 18,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})


In [ ]:
folderVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/Tests/RotatingMonopole/monopole/probes/probe1",
    "/Users/maggie/repo/LBM-Program/amrlbm/Tests/RotatingMonopole/dipole/probes/probe1",
]


labelVec = [
    "Present LBM D2Q9 Regularized-BGK",
]

colorVec = [
    "#05A361",
    "#D33737",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

markerVec = [
    'o',
    '^',
    's',
    "P",
    '*',
]

freq = 0.04
# M = [ 0.001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 ] # monopole
M = [ 0.0, 0.001, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0 ] # dipole-x
vis = 0.0
rhoRef = 1.0

Lx = 201
Ly = 201

step = 400

c_s = 1.0 / np.sqrt(3.0)
omega = 2 * np.pi * freq
k = omega / c_s


In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec
def get_probe_coord_value(filePath, coordIdx):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordValue = float(header.split('(')[1].split(')')[0].split(',')[coordIdx])
    return coordValue



In [ ]:

coordMap = { 'x': 0, 'y': 1, 'z': 2 }
coordIdx = coordMap['x']
colIdx = 2

coordVecVec = []
dataVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderVec[case]}/probe*.txt"))
    coordVec = []
    dataVec = []
    for filePath in fileList:
        coordValue = get_probe_coord_value(filePath, coordIdx)
        coordVec.append(coordValue)
        stepCol = read_col_probe(filePath, 0, 2)
        dataCol = read_col_probe(filePath, colIdx)
        targetStepIdx = int(np.abs(stepCol - (step + 1)).argmin())
        dataVec.append((dataCol[targetStepIdx] - rhoRef) * c_s**2)
    coordVecVec.append(coordVec)
    dataVecVec.append(dataVec)


In [ ]:

#! analytical solution
x = np.linspace(0.0, 100, 300)
xSafe = np.where(x == 0, 1, x)

wt = [ 4.0 / 9.0, 1.0 / 9.0, 1.0 / 9.0, 1.0 / 9.0, 1.0 / 9.0, 1.0 / 36, 1.0 / 36, 1.0 / 36, 1.0 / 36 ]
cx = [ 0, 1, -1,  0,  0,  1, -1,  1, -1 ]
cy = [ 0, 0,  0,  1, -1,  1,  1, -1, -1 ]
xPyP = 1.0 / np.sqrt(8)
src = np.zeros(9)
src[0] = (wt[0] * M[0] - M[3] - M[4] - 0.5 * M[7] - 0.5 * M[8])
src[1] = (wt[1] * M[0] + 0.5 * M[1] + 0.5 * M[3])
src[2] = (wt[2] * M[0] - 0.5 * M[1] + 0.5 * M[3])
src[3] = (wt[3] * M[0] + 0.5 * M[2] + 0.5 * M[4])
src[4] = (wt[4] * M[0] - 0.5 * M[2] + 0.5 * M[4])
src[5] = (wt[5] * M[0] + xPyP * M[5] + 0.25 * M[7])
src[6] = (wt[6] * M[0] + xPyP * M[6] + 0.25 * M[8])
src[7] = (wt[7] * M[0] - xPyP * M[5] + 0.25 * M[7])
src[8] = (wt[8] * M[0] - xPyP * M[6] + 0.25 * M[8])
# print(src)
S0 = 0.0
Sx = 0.0
Sy = 0.0
for i in range(0, 9):
    S0 += src[i]
    Sx += src[i] * cx[i]
    Sy += src[i] * cy[i]

print(S0, Sx, Sy)

Gw = 1.0 / (4 * 1j) * hankel2(0, k * np.abs(x)) * np.exp(1j * omega * step) * -1j #! sin
dx = 1
# dGwDx = 1.0 / (4 * 1j) * (hankel2(0, k * np.abs(x + dx)) - hankel2(0, k * np.abs(x))) / dx * np.exp(1j * omega * step) * -1j #! sin
dGwDx = 1.0 / (4 * 1j) * (-k * hankel2(1, k * np.abs(x))) * np.sign(x) * np.exp(1j * omega * step) * -1j
pPrimeRef = 1j * omega * S0 * Gw - Sx * dGwDx


In [ ]:

fig, ax = plt.subplots(1, 1, figsize=(10, 4), facecolor='w', edgecolor='w')

ax.plot(xSafe, np.real(pPrimeRef), c='k', lw=1.5, label='Analytical solution')
for case in range(0, len(folderVec)):
    ax.plot(coordVecVec[case], dataVecVec[case], c=colorVec[case], marker=markerVec[case], fillstyle='none', lw=0, mew=1.5, markevery=1, label=labelVec[case])

ax.set_ylim(-0.0001, 0.0001)
ax.set_xlim(0, 80)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$p\prime$")
ax.legend(loc='upper right', frameon=False)
